In [1]:
## Installation required
## In qbraid we need to run the cell everytime we open a new session

!pip install qiskit[visualization]
# Use the following if you are on MacOS/zsh
#!pip install 'qiskit[visualization]'==1.1.0
!pip install qiskit_ibm_runtime
!pip install qiskit_aer
!pip install matplotlib
!pip install pylatexenc
!pip install prototype-zne

In [2]:
import qiskit
qiskit.__version__

'2.6.2'

In [3]:
## Import packages

#from math import pi
#import numpy as np
import time
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
#from qiskit.circuit import Gate
from qiskit_aer import AerSimulator, QasmSimulator
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.visualization import plot_histogram

size = 5*32*64 #nb*h*k = #10240 (nb=5) or 2048 (nb=1)
V = QuantumRegister(size) #10240 or 2048
X = QuantumRegister(size) #10240 or 2048
qr = QuantumRegister(288)
t = QuantumRegister(3)
cr = ClassicalRegister(2*size+288+3) #128*2+288+3=547
qc = QuantumCircuit(V, X, qr, t, cr)

In [4]:
num_qubits = qc.num_qubits
print("Number of qubits:", num_qubits)

Number of qubits: 20771


In [5]:
def swap_to_cx(qc, q1, q2):
    qc.cx(q1, q2)
    qc.cx(q2, q1)
    qc.cx(q1, q2)

In [6]:
key = '000000010010001101000101011001111000'+'0000'+'1101010111100110111101111000100100011010'
#key = '00000001001000110100010101100111100010011010101111001101111011110001001000110100' 
iv = '00000001001000110100010101100111100010011010101111001101111011110001001000110100'  
for i in range(80): #For loading Key
    if (key[i] == "1"):
        qc.x(qr[i]) 
for i in range(80): #For loading IV
    if (iv[i] == "1"):
        qc.x(qr[i+93]) 

qc.x(qr[285])
qc.x(qr[286])
qc.x(qr[287])

def key_gen2(qc, qr, V, round, ks):
    for i in range(round):        
        qc.cx(qr[65],qr[92])
        qc.ccx(qr[90],qr[91],qr[92])
        qc.cx(qr[170],qr[92])
        swap_to_cx(qc, qr[92],t[0])
        
        qc.cx(qr[161],qr[176])
        qc.ccx(qr[174],qr[175],qr[176])
        qc.cx(qr[263],qr[176])
        swap_to_cx(qc, qr[176],t[1])
        
        qc.cx(qr[242],qr[287])
        qc.ccx(qr[285],qr[286],qr[287])
        qc.cx(qr[68],qr[287])
        swap_to_cx(qc, qr[287],t[2])

        for j in range(287,176,-1):
            if j == 177:
                swap_to_cx(qc, qr[177], t[1])
            else:
                swap_to_cx(qc, qr[j], qr[j-1])

        for j in range(176,92,-1):
            if j == 93:
                swap_to_cx(qc, qr[93], t[0])
            else:
                swap_to_cx(qc, qr[j], qr[j-1])
        
        for j in range(92,-1,-1):
            if j == 0:
                swap_to_cx(qc, qr[0], t[2])
            else:
                swap_to_cx(qc, qr[j], qr[j-1])


    """ Key Stream Generation """
    for i in range(ks):
        
        qc.cx(qr[65],qr[92])
        swap_to_cx(qc, qr[92],t[0])
        
        qc.cx(qr[161],qr[176])
        swap_to_cx(qc, qr[176],t[1])
        
        qc.cx(qr[242],qr[287])
        swap_to_cx(qc, qr[287],t[2])
        
        qc.cx(t[0],V[i])
        qc.cx(t[1],V[i])
        qc.cx(t[2],V[i])
        
        qc.ccx(qr[90],qr[91],t[0]) 
        qc.cx(qr[170],t[0]) 
        
        qc.ccx(qr[174],qr[175],t[1]) 
        qc.cx(qr[263],t[1]) 
        
        qc.ccx(qr[285],qr[286],t[2]) 
        qc.cx(qr[68],t[2]) 
        
        for j in range(287,176,-1):
            if j == 177:
                swap_to_cx(qc, qr[177], t[1])
            else:
                swap_to_cx(qc, qr[j], qr[j-1])

        for j in range(176,92,-1):
            if j == 93:
                swap_to_cx(qc, qr[93], t[0])
            else:
                swap_to_cx(qc, qr[j], qr[j-1])
        
        for j in range(92,-1,-1):
            if j == 0:
                swap_to_cx(qc, qr[0], t[2])
            else:
                swap_to_cx(qc, qr[j], qr[j-1])
                


In [7]:
def algo1_org(qc, V, X):
    nb = 5; delta = 64; H = 32; W = 64
    pi1=[2 for i in range(H)]; pi2=[2 for i in range(H)]  #some dummy values
    for j in range(1, nb+1):
        if j%delta == 0:
            algo2_org(j, pi1, pi2, H, W)
        for h in range(1, H+1):
            for k in range(1, W+1):
                index_k = (h - 1) * W + k
                index1 = (j - 1) * H * W + (h - 1) * W + k
                index2 = (j - 1) * H * W + (pi1[h-1] - 1) * W + k
                swap_to_cx(qc, V[index1-1], X[index_k-1])
                qc.cx(V[index2-1], X[index_k-1])
                index3 = (j - 1) * H * W + (pi2[h-1] - 1) * W + k
                qc.cx(V[index3-1], X[index_k-1])
        
        for h in range(1, H+1):
            for k in range(1, W+1):
                index_k = (h - 1) * W + k
                index_k1 = (h - 1) * W + (k - 2) % W #in place of 12
                qc.cx(X[index_k1-1], X[index_k-1])
                index_k2 = (h - 1) * W + (k + 5) % W #in place of 25
                index_k3 = (h - 1) * W + (k - 7) % W #in place of 27
                qc.cx(X[index_k2-1], X[index_k-1])
                qc.cx(X[index_k3-1], X[index_k-1])
                swap_to_cx(qc, V[((j-1) * H * W + index_k)-1], X[index_k-1])
                
        for h in range(1, H+1):
            for k in range(1, W+1):
                index_x = (h - 1) * W + k
                index1 = (j - 1) * H * W + (h-1) * W + k
                index2 = (j - 1) * H * W + ((h - pi1[h-1]) % W) + k #in place of (j - 1) * H * W + ((h - pi1[h-1]) % W - 1) * W + k
                swap_to_cx(qc, V[index2-1], X[index_x-1])
                swap_to_cx(qc, V[index1-1], X[index_x-1])
                swap_to_cx(qc, V[index2-1], X[index_x-1])
                

def algo2_org(j, pi1, pi2, H, W):
    for h in range(1, H+1):
        for k in range(1, W+1):
            index_x = (h - 1) * W + k
            index1 = (j - 1) * H * W + (h-1) * W + k
            index2 = (j - 1) * H * W + (pi1[h-1] - 1) * W + k
            index3 = (j - 1) * H * W + (pi2[h-1] - 1) * W + k
            qc.ccx(V[index1-1], X[index2-1], X[index_x-1])
            qc.cx(V[index3-1], X[index_x-1])
            swap_to_cx(qc, V[index1-1], X[index_x-1])

In [8]:
#start = time.perf_counter()
key_gen2(qc, qr, V, 1152, 10240) # This is a part of initialization
algo1_org(qc, V, X)

In [9]:
# Define a Clifford+T gate basis
clifford_t_basis = [
    'h', 's', 'sdg',
    'x', 'y', 'z',
    'cx', 't', 'tdg'
]

# Decomposiion of the entire circuit into the selected basis
qc_ct = transpile(
    qc,
    basis_gates=clifford_t_basis,
    optimization_level=0
)
# optimization_level=0 here to avoid introducing additional optimization transformations while decomposing the original circuit

# Full depth of the decomposed circuit
full_depth_ct = qc_ct.depth()

print("Original Full Depth:", qc.depth())
print("Clifford+T Full Depth:", full_depth_ct)

Original Full Depth: 3812039
Clifford+T Full Depth: 3914438
